# Práctica Final - Versión Moderna (LCEL)
### Automatización de respuestas a emails de devolución
**Componentes Intergalácticos Industriales S.A.**

Esta versión utiliza:
- `PydanticOutputParser` → parseo estructurado
- Sintaxis LCEL con `|` → sin LLMChain ni SequentialChain

## 1. Instalación de dependencias

In [1]:
%pip install langchain langchain-groq python-dotenv pydantic -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\macdu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Imports y configuración

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()

# Opción A (recomendado): crea un archivo .env con: GROQ_API_KEY=gsk_tu-clave-aqui
# Opción B: descomenta la siguiente línea
# os.environ['GROQ_API_KEY'] = 'gsk_tu-clave-aqui'

llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0)


## 3. Parser de salida estructurada
Definimos con Pydantic la estructura esperada.
El parser genera automáticamente las instrucciones de formato para el prompt.

In [3]:
class InfoEmail(BaseModel):
    pedido: str = Field(description='Número de pedido')
    remitente: str = Field(description='Nombre del remitente')
    motivo: str = Field(description='Motivo principal de la solicitud')

parser_info = PydanticOutputParser(pydantic_object=InfoEmail)


## 4. Paso 1 — Extracción de información
Sintaxis moderna: `prompt | llm | parser` en lugar de `LLMChain`.

In [4]:
extract_prompt = PromptTemplate(
    input_variables=['email'],
    partial_variables={'format_instructions': parser_info.get_format_instructions()},
    template="""Extrae la siguiente información del email:
- Número de pedido
- Nombre del remitente
- Motivo principal de la solicitud

{format_instructions}

Email:
{email}
"""
)

# Sintaxis moderna LCEL: encadenamos con |
extract_chain = extract_prompt | llm | parser_info


## 5. Paso 2 — Evaluación de la solicitud
Recibe el motivo del paso anterior y decide ACEPTAR o RECHAZAR.

In [5]:
eval_prompt = PromptTemplate(
    input_variables=['motivo'],
    template="""Según este motivo: "{motivo}", decide si se debe ACEPTAR o RECHAZAR la devolución.

ACEPTAR si:
- Defecto de fabricación
- Error en el suministro
- Producto incompleto desde fábrica

RECHAZAR si:
- Daños durante el transporte (si no es responsabilidad de la empresa)
- Manipulación del cliente
- Solicitud fuera de plazo

Responde ÚNICAMENTE con: ACEPTAR o RECHAZAR
"""
)

eval_chain = eval_prompt | llm | StrOutputParser()


## 6. Paso 3 — Redacción de la respuesta
Recibe remitente, pedido y decisión y redacta el email de respuesta.

In [6]:
response_prompt = PromptTemplate(
    input_variables=['remitente', 'pedido', 'decision'],
    template="""Redacta una respuesta formal y empática para el cliente {remitente}, sobre el pedido {pedido}.

Si la decisión es ACEPTAR:
- Agradécele por contactar.
- Confirma que el reemplazo será procesado.
- Explica que la empresa cubrirá el fallo según la política.

Si la decisión es RECHAZAR:
- Lamenta la situación.
- Explica que no se puede aceptar la devolución según la política.
- Ofrece ayuda adicional si la necesita.

Finaliza con esta firma:
María Fernández
Responsable de Atención al Cliente
Componentes Intergalácticos Industriales S.A.
contacto@cii.com

Decisión: {decision}
"""
)

response_chain = response_prompt | llm | StrOutputParser()


## 7. Función principal
Ejecuta los 3 pasos en secuencia pasando las variables entre ellos manualmente.
Con LCEL no necesitamos SequentialChain, el flujo es explícito y legible.

In [7]:
def ejecutar_cadena(email):
    # Paso 1: extraer info → devuelve objeto InfoEmail
    info = extract_chain.invoke({'email': email})

    # Paso 2: evaluar → devuelve 'ACEPTAR' o 'RECHAZAR'
    decision = eval_chain.invoke({'motivo': info.motivo}).strip()

    # Paso 3: redactar respuesta
    respuesta = response_chain.invoke({
        'remitente': info.remitente,
        'pedido': info.pedido,
        'decision': decision
    })

    return {'info': info, 'decision': decision, 'respuesta': respuesta}


## 8. Caso 1 — Solicitud RECHAZADA (daños en transporte)

In [8]:
email_rechazar = """Asunto: Solicitud de reemplazo por daños en transporte - Pedido #D347-STELLA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Me pongo en contacto con ustedes para comunicar una incidencia relacionada con el
pedido #D347-STELLA, correspondiente a un lote de condensadores de fluzo modelo FX-88.

Al recibir el envío, varios condensadores presentaban daños visibles. Todo indica que
la mercancía sufrió una caída durante el transporte interestelar.

Solicitamos con urgencia el reemplazo inmediato de las unidades defectuosas.

Atentamente,
Darth Márquez
"""

resultado1 = ejecutar_cadena(email_rechazar)

print('=' * 60)
print('CASO 1 - RECHAZAR')
print('=' * 60)
print('Info extraída:', resultado1['info'])
print('Decisión:', resultado1['decision'])
print('Respuesta:\n', resultado1['respuesta'])


CASO 1 - RECHAZAR
Info extraída: pedido='D347-STELLA' remitente='Darth Márquez' motivo='Reemplazo por daños en transporte'
Decisión: RECHAZAR
Respuesta:
 Estimado Sr. Darth Márquez:

Lamentamos profundamente la situación que ha experimentado con el pedido **D347‑STELLA** y agradecemos que nos haya puesto en conocimiento de su caso.

Tras revisar detenidamente la solicitud y la normativa interna de nuestra empresa, debemos informarle que, según la política de devoluciones vigente, no nos es posible aceptar la devolución del producto en las condiciones presentadas. Entendemos que esta respuesta puede resultar decepcionante y le pedimos disculpas por cualquier inconveniente que ello le pueda ocasionar.

Quedamos a su disposición para brindarle la asistencia que necesite respecto a este asunto o cualquier otra consulta que pueda tener. No dude en contactarnos y con gusto le ofreceremos el apoyo necesario.

Atentamente,

María Fernández  
Responsable de Atención al Cliente  
Componentes Int

## 9. Caso 2 — Solicitud ACEPTADA (defecto de fabricación)

In [9]:
email_aceptar = """Asunto: Devolución por defecto de fabricación - Pedido #XZ901-LUCA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Les escribo para informarles que el pedido #XZ901-LUCA, compuesto por microprocesadores
CU-92, presenta un defecto de fábrica: varias unidades no responden a la activación básica.

Solicito formalmente la devolución o el reemplazo de las unidades defectuosas.

Atentamente,
Lucía Robles
"""

resultado2 = ejecutar_cadena(email_aceptar)

print('=' * 60)
print('CASO 2 - ACEPTAR')
print('=' * 60)
print('Info extraída:', resultado2['info'])
print('Decisión:', resultado2['decision'])
print('Respuesta:\n', resultado2['respuesta'])


CASO 2 - ACEPTAR
Info extraída: pedido='XZ901-LUCA' remitente='Lucía Robles' motivo='Devolución o reemplazo de unidades defectuosas por defecto de fabricación'
Decisión: ACEPTAR
Respuesta:
 Estimada Sra. Lucía Robles,

Muchas gracias por ponerse en contacto con nosotros respecto al pedido **XZ901‑LUCA**. Lamentamos los inconvenientes que ha experimentado y le agradecemos la oportunidad de poder asistirla.

Nos complace informarle que hemos **aceptado su solicitud de reemplazo**. El proceso de sustitución se iniciará de inmediato y, conforme a nuestra política de garantía, **la empresa cubrirá íntegramente el fallo del producto**, sin coste adicional para usted.

En los próximos días recibirá un correo de confirmación con los detalles del envío y la fecha estimada de entrega. Si necesita cualquier información adicional o tiene alguna otra consulta, no dude en comunicarse con nosotros; estaremos encantados de ayudarle.

Quedamos a su disposición para cualquier otra necesidad que pueda su